# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 个人GitHub项目说明

- 每名学生独立完成本Notebook；
- 输入文件固定为`data/E Commerce Dataset.xlsx`；
- 输出固定写入`output/day04_project/`；
- 不要提交教师演示Notebook或教师参考答案；
- 完成后重启内核并从头运行，再推送到个人GitHub仓库。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [46]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")


def find_project_root(start=None):
    """从当前目录向上寻找种子项目根目录。"""
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "E Commerce Dataset.xlsx").exists():
            return candidate
    raise FileNotFoundError("未找到data/E Commerce Dataset.xlsx")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "E Commerce Dataset.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
raw_df.head()

原始数据：c:\Users\l\Desktop\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\data\E Commerce Dataset.xlsx
项目输出目录：c:\Users\l\Desktop\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project
原始数据形状：(5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [47]:
# 答案：
# 1. 每条记录对应一位电商平台的独立用户（客户），存储了该用户的唯一 ID、流失状态、在平台的使用时长、登录偏好、所在城市等级、配送相关信息、支付偏好、性别等多维度的个人与行为数据。
# 2. 目标变量是 “Churn（用户流失）”列：它是二分类标识（1 代表用户流失、0 代表用户留存），这类用户流失分析项目的核心目标通常就是预测 / 分析用户是否流失，所以Churn是要研究、预测的目标变量。
# 3. CustomerID 是仅用于区分用户的名义编号，数值大小无业务数量含义，无法进行连续数值类的统计建模运算，因此不能作为普通连续数值分析

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [48]:
def build_quality_report(data):
    """返回字段级数据质量报告。"""
    # 计算各字段的指标
    dtypes = data.dtypes
    missing_count = data.isnull().sum()
    missing_pct = (missing_count / len(data) * 100).round(2)
    unique_count = data.nunique()
    
    # 构建成 DataFrame
    report = pd.DataFrame({
        "数据类型": dtypes,
        "缺失数量": missing_count,
        "缺失比例(%)": missing_pct,
        "唯一值数量": unique_count
    })
    
    return report

# 生成清洗前质量报告
quality_before = build_quality_report(raw_df)
display(quality_before)

,数据类型,缺失数量,缺失比例(%),唯一值数量
CustomerID,int64,0,0.00,5630
Churn,int64,0,0.00,2
Tenure,float64,264,4.69,36
PreferredLoginDevice,str,0,0.00,3
CityTier,int64,0,0.00,3
WarehouseToHome,float64,251,4.46,34
PreferredPaymentMode,str,0,0.00,7
Gender,str,0,0.00,2
HourSpendOnApp,float64,255,4.53,6
NumberOfDeviceRegistered,int64,0,0.00,6


### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [49]:
# TODO：完成项目初始审计
# 1. 完全重复行数
duplicate_rows = raw_df.duplicated().sum()
print("完全重复行数：", duplicate_rows)

# 2. CustomerID 重复数量
duplicate_customerid = raw_df["CustomerID"].duplicated().sum()
print("CustomerID 重复数量：", duplicate_customerid)

# 3. Churn 的频数和流失率
print("\n=== Churn 分布 ===")
churn_counts = raw_df["Churn"].value_counts()
print(churn_counts)

churn_rate = raw_df["Churn"].mean() * 100
print("流失率：{:.2f}%".format(churn_rate))

# 4. 主要类别字段的频数
print("\n=== 主要类别字段频数 ===")
for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"\n--- {col} ---")
    print(raw_df[col].value_counts())

完全重复行数： 0
CustomerID 重复数量： 0

=== Churn 分布 ===
Churn
0    4682
1     948
Name: count, dtype: int64
流失率：16.84%

=== 主要类别字段频数 ===

--- PreferredLoginDevice ---
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

--- PreferredPaymentMode ---
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

--- PreferedOrderCat ---
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [50]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [51]:

    # TODO：复制数据，避免覆盖原始数据
    # TODO：创建日志列表 logs
    # TODO：删除完全重复行，并记录日志
    # TODO：对 NUMERIC_MISSING_COLS 使用中位数填补，并记录每列影响数量
    # TODO：对 CATEGORY_MAPPINGS 完成类别标准化，并记录每条映射影响数量
    # TODO：将 Churn 和 Complain 转为整数类型
    # TODO：返回 cleaned_df 与 cleaning_log
# 按你给的规则，先定义清洗规则变量
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。
    """
    df = data.copy()
    logs = []
    initial_count = len(df)

    # 1. 删除重复行
    before = len(df)
    df = df.drop_duplicates()
    logs.append({
        "处理步骤": "删除重复记录",
        "处理规则": "删除完全重复的整行记录",
        "处理前记录数": before,
        "处理后记录数": len(df),
        "影响记录数": before - len(df)
    })

    # 2. 数值列缺失值处理（强制生效版）
    NUMERIC_MISSING_COLS = [
        "Tenure",
        "WarehouseToHome",
        "HourSpendOnApp",
        "OrderAmountHikeFromlastYear",
        "CouponUsed",
        "OrderCount",
        "DaySinceLastOrder"
    ]
    
    for col in NUMERIC_MISSING_COLS:
        if col in df.columns:
            # 关键：先强制转成数值类型，再用中位数填充
            df[col] = pd.to_numeric(df[col], errors='coerce')
            med = df[col].median()
            df[col] = df[col].fillna(med)
            logs.append({
                "处理步骤": "缺失值填充",
                "处理规则": f"列 {col} 使用总体中位数填充",
                "处理前记录数": initial_count,
                "处理后记录数": len(df),
                "影响记录数": df[col].isna().sum()
            })

    # 3. 类别标准化
    CATEGORY_MAPPINGS = {
        "PreferredLoginDevice": {
            "Phone": "Mobile Phone",
            "Mobile": "Mobile Phone"
        },
        "PreferredPaymentMode": {
            "COD": "Cash on Delivery",
            "CC": "Credit Card"
        },
        "PreferedOrderCat": {
            "Mobile": "Mobile Phone"
        }
    }
    
    for col, mapping in CATEGORY_MAPPINGS.items():
        if col in df.columns:
            df[col] = df[col].replace(mapping)

    # 4. 类型转换
    if "Churn" in df.columns:
        df["Churn"] = df["Churn"].astype(int)
    if "Complain" in df.columns:
        df["Complain"] = df["Complain"].astype(int)

    cleaning_log = pd.DataFrame(logs)
    return df, cleaning_log

In [52]:
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)
assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0, "数值列仍存在缺失值"
print("✅ 断言通过！")

✅ 断言通过！


### 任务 3：运行清洗函数并查看日志

In [53]:
# TODO：执行清洗
# 调用清洗函数
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

# 查看日志
display(cleaning_log)

# 查看清洗后的数据
display(cleaned_df.head())

,处理步骤,处理规则,处理前记录数,处理后记录数,影响记录数
0,删除重复记录,删除完全重复的整行记录,5630,5630,0
1,缺失值填充,列 Tenure 使用总体中位数填充,5630,5630,0
2,缺失值填充,列 WarehouseToHome 使用总体中位数填充,5630,5630,0
3,缺失值填充,列 HourSpendOnApp 使用总体中位数填充,5630,5630,0
4,缺失值填充,列 OrderAmountHikeFromlastYear 使用总体中位数填充,5630,5630,0
5,缺失值填充,列 CouponUsed 使用总体中位数填充,5630,5630,0
6,缺失值填充,列 OrderCount 使用总体中位数填充,5630,5630,0
7,缺失值填充,列 DaySinceLastOrder 使用总体中位数填充,5630,5630,0


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [54]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }

# TODO：构建 tenure_bins、tenure_labels，并用 pd.cut 新建 TenureGroup
# TODO：新建 IsMobileLogin，移动端为 1，其他设备为 0
# TODO：生成 outlier_report（每行对应一个待检查字段）
# 1. 构建 TenureGroup：用户使用时长分层
# 这里按常见的电商用户分层逻辑，分成4个区间，可根据实际数据调整
tenure_bins = [-1, 0, 12, 24, float('inf')]
tenure_labels = ['0个月', '1-12个月', '13-24个月', '24个月以上']
cleaned_df['TenureGroup'] = pd.cut(cleaned_df['Tenure'], 
                                   bins=tenure_bins, 
                                   labels=tenure_labels)

# 2. 新建 IsMobileLogin：是否主要使用移动端登录
# 假设 PreferredLoginDevice 列中，"Mobile" 或 "Phone" 代表移动端
cleaned_df['IsMobileLogin'] = cleaned_df['PreferredLoginDevice'].isin(['Mobile', 'Mobile Phone', 'Phone']).astype(int)

# 3. 生成候选异常值报告（IQR 方法）
# 待检查的字段列表
check_cols = ["WarehouseToHome", "OrderCount", "CashbackAmount"]

outlier_report_list = []
for col in check_cols:
    if col in cleaned_df.columns:
        result = iqr_outlier_summary(cleaned_df[col])
        outlier_report_list.append({
            "字段名": col,
            "Q1": result["Q1"],
            "Q3": result["Q3"],
            "下限": result["下限"],
            "上限": result["上限"],
            "候选异常值数量": result["候选异常值数量"]
        })

outlier_report = pd.DataFrame(outlier_report_list)

# 查看结果
display(cleaned_df[['Tenure', 'TenureGroup', 'PreferredLoginDevice', 'IsMobileLogin']].head())
display(outlier_report)

,Tenure,TenureGroup,PreferredLoginDevice,IsMobileLogin
0,4.00,1-12个月,Mobile Phone,1
1,9.00,1-12个月,Mobile Phone,1
2,9.00,1-12个月,Mobile Phone,1
3,0.00,0个月,Mobile Phone,1
4,0.00,0个月,Mobile Phone,1


,字段名,Q1,Q3,下限,上限,候选异常值数量
0,WarehouseToHome,9.00,20.00,-7.50,36.50,2
1,OrderCount,1.00,3.00,-2.00,6.00,703
2,CashbackAmount,145.77,196.39,69.84,272.33,438


### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [55]:
# TODO：完成业务规则检查
# business_rule_report = pd.DataFrame({
#     "规则": [...],
#     "不合规记录数": [...]
# })
# display(business_rule_report)
#
# 业务规则检查
# 1. 使用时长小于 0
invalid_tenure = (cleaned_df["Tenure"] < 0).sum()
# 2. 仓库距离小于 0
invalid_warehouse = (cleaned_df["WarehouseToHome"] < 0).sum()
# 3. 订单数小于或等于 0
invalid_ordercount = (cleaned_df["OrderCount"] <= 0).sum()
# 4. 返现金额小于 0
invalid_cashback = (cleaned_df["CashbackAmount"] < 0).sum()

# 构建报告 DataFrame
business_rule_report = pd.DataFrame({
    "规则": [
        "使用时长小于 0",
        "仓库距离小于 0",
        "订单数小于或等于 0",
        "返现金额小于 0"
    ],
    "不合规记录数": [
        invalid_tenure,
        invalid_warehouse,
        invalid_ordercount,
        invalid_cashback
    ]
})

display(business_rule_report)
# 处理结论：规则 1（使用时长小于 0）：本次检查中，不合规记录数为 X 条。由于用户使用时长不可能为负数，这类记录属于数据录入或系统错误，应在清洗中直接删除。
#规则 2（仓库距离小于 0）：不合规记录数为 X 条。物理距离无法为负，此类记录属于异常值，应直接删除。
#规则 3（订单数≤0）：不合规记录数为 X 条。订单数为 0 的用户属于 “零行为用户”，无法反映用户行为模式；订单数为负属于数据错误，这两类记录均应直接删除。
#规则 4（返现金额小于 0）：不合规记录数为 X 条。返现金额为负意味着用户倒贴平台，不符合业务逻辑，应直接删除

,规则,不合规记录数
0,使用时长小于 0,0
1,仓库距离小于 0,0
2,订单数小于或等于 0,0
3,返现金额小于 0,0


---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [56]:
# TODO：完成最终验收
# quality_after = build_quality_report(cleaned_df)
#
# assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0
# assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique()
# assert "COD" not in cleaned_df["PreferredPaymentMode"].unique()
# assert "CC" not in cleaned_df["PreferredPaymentMode"].unique()
# assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns)
#
# TODO：导出下列文件，使用 utf-8-sig 编码：
# quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=True, encoding="utf-8-sig")
# quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=True, encoding="utf-8-sig")
# cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
# cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")
#
# TODO：输出 outlier_report 和 business_rule_report
# TODO：输出交付文件的路径
# --- 1. 完成最终验收 ---
# 生成清洗后数据质量报告
quality_after = build_quality_report(cleaned_df)

# 项目验收断言检查
assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0, "数值列仍存在缺失值"
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique(), "PreferredLoginDevice 仍存在未标准化的值 'Phone'"
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique(), "PreferredPaymentMode 仍存在未标准化的值 'COD'"
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique(), "PreferredPaymentMode 仍存在未标准化的值 'CC'"
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns), "衍生字段 TenureGroup 或 IsMobileLogin 缺失"

print("✅ 所有验收断言检查通过！")

# --- 2. 导出下列文件，使用 utf-8-sig 编码 ---
# 清洗前后数据质量报告
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=True, encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=True, encoding="utf-8-sig")

# 清洗日志
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")

# 清洗后的数据
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")

# --- 3. 输出 outlier_report 和 business_rule_report ---
outlier_report.to_csv(OUTPUT_DIR / "outlier_report.csv", index=False, encoding="utf-8-sig")
business_rule_report.to_csv(OUTPUT_DIR / "business_rule_report.csv", index=False, encoding="utf-8-sig")

# --- 4. 输出交付文件的路径 ---
print("\n📁 所有交付物已导出到以下路径：")
for file in OUTPUT_DIR.glob("*.csv"):
    print(f"- {file.resolve()}")

✅ 所有验收断言检查通过！

📁 所有交付物已导出到以下路径：
- C:\Users\l\Desktop\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\business_rule_report.csv
- C:\Users\l\Desktop\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\cleaning_log.csv
- C:\Users\l\Desktop\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\data_quality_after.csv
- C:\Users\l\Desktop\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\data_quality_before.csv
- C:\Users\l\Desktop\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\ecommerce_customer_cleaned.csv
- C:\Users\l\Desktop\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\outlier_report.csv


## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？

   存在数值字段缺失、类别字段命名不统一、候选异常值三类问题。
2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？

   缺失值用总体中位数填补；不一致类别按业务口径统一命名；候选异常值仅记录留存，不直接删除待复核。
3. 为什么清洗后的数据可以作为第五天分析的输入？

   清洗后消除了重复、缺失、类别混乱问题，格式规范、口径统一，无业务不合规值，满足后续分析的数据基础要求。
4. 哪些处理规则仍需要业务人员确认？

   候选异常值的判定边界、异常记录最终保留 / 剔除方案，需要业务人员结合实际场景确认。

## GitHub提交检查

- [ ] Notebook已重启内核并从头运行成功；
- [ ] `output/day04_project/`包含清洗后数据、质量报告、清洗日志和异常/业务规则报告；
- [ ] 原始Excel没有被覆盖；
- [ ] 清洗函数、处理日志和项目复盘均已完成；
- [ ] 已提交并推送到个人GitHub仓库。